In [1]:
# In this script we select demographic/socioeconomic variables of interest
# pre-compute their shares of total population, and join attribute and spatial CBG-Level data

In [2]:
import os
import pandas as pd
import geopandas as gpd

In [3]:
# Configurations

datadir = '/projects/standard/lenkne/oboiko/EJ/data/'
outdir = datadir + '/processed/population/'
if not os.path.exists(outdir):
    os.makedirs(outdir)
out_filepath = outdir + 'NHGIS_cbg_MissRiv_{}.gpkg'

data_dictionary = {
    '2008_2012' : {
        'boundary_filepath': 'US_blck_grp_2012.shp',
        'table_filepath': 'nhgis0058_ds191_20125_blck_grp.csv',
        'code_race': 'QSQ',
        'code_ethnicity': 'QSY',
        'code_poverty' : 'QUV'
    },
    '2013_2017' : {
        'boundary_filepath': 'US_blck_grp_2017.shp',
        'table_filepath': 'nhgis0057_ds233_20175_blck_grp.csv',
        'code_race': 'AHY2',
        'code_ethnicity': 'AHZA',
        'code_poverty' : 'AH1J'
    },
     '2018_2022' : {
        'boundary_filepath': 'US_blck_grp_2022.shp',
        'table_filepath': 'nhgis0056_ds262_20225_blck_grp.csv',
        'code_race': 'AQNG',
        'code_ethnicity': 'AQNO',
        'code_poverty' : 'AQPZ'
    }
}

# 10 states of interest - those intersect Mississippi River
state_ids = ['05', '17', '19', '21', '22', '27', '28', '29', '47', '55'] 

In [4]:
for study_period, metadata in data_dictionary.items():
    print ('\nProcessing', study_period,  metadata)
    boundary_filepath = metadata['boundary_filepath']
    table_filepath = metadata['table_filepath']
    CR = metadata['code_race']
    CE = metadata['code_ethnicity']
    CP = metadata['code_poverty']
    # open national shapefile
    cbg_gdf = gpd.read_file(datadir + f'raw/population/{boundary_filepath}')
    # select 10 states of interest
    cbg_gdf_selected = cbg_gdf[cbg_gdf['STATEFP'].isin(state_ids)]
    print (f'Count of all CBGs in the US: {len(cbg_gdf)}')
    print (f'Count of CBGs in the 10 states of interest: {len(cbg_gdf_selected)}')
    # open table
    cbg_table = pd.read_csv(datadir + f'raw/population/{table_filepath}')
    print (f'CBG table records: {len(cbg_table)}')
    # perform join to combine spatial and attribute info
    joined = cbg_gdf_selected.merge(cbg_table, on='GISJOIN')
    print (f'Count of CBGs in the study area after join: {len(joined)}')
    print ('Compute race and ethnicity shares')
    joined['total'] = joined[f'{CR}E001']
    joined['share_black'] = joined[f'{CR}E003']/joined[f'{CR}E001']
    joined['share_native'] = joined[f'{CR}E004']/joined[f'{CR}E001']
    joined['share_asian'] = joined[f'{CR}E005']/joined[f'{CR}E001']
    joined['share_hispanic'] = joined[f'{CE}E012']/joined[f'{CE}E001']
    joined['share_nonhsp_white'] = joined[f'{CE}E003']/joined[f'{CE}E001']
    print ('Compute poverty shares')
    joined['share_2_below_poverty'] = joined[f'{CP}E002']/joined[f'{CP}E001']
    joined['share_2_above_poverty'] = joined[f'{CP}E008']/joined[f'{CP}E001']
    joined['share_below_poverty'] = (joined[f'{CP}E002']+joined[f'{CP}E003'])/joined[f'{CP}E001']
    print ('Saving to a file')
    joined.to_file(out_filepath.format(study_period))


Processing 2008_2012 {'boundary_filepath': 'US_blck_grp_2012.shp', 'table_filepath': 'nhgis0058_ds191_20125_blck_grp.csv', 'code_race': 'QSQ', 'code_ethnicity': 'QSY', 'code_poverty': 'QUV'}


ERROR 1: PROJ: proj_create_from_database: Open of /users/2/oboiko/.conda/envs/geo/share/proj failed


Count of all CBGs in the US: 219774
Count of CBGs in the 10 states of interest: 40587
CBG table records: 220333
Count of CBGs in the study area after join: 40587
Compute race and ethnicity shares
Compute poverty shares
Saving to a file

Processing 2013_2017 {'boundary_filepath': 'US_blck_grp_2017.shp', 'table_filepath': 'nhgis0057_ds233_20175_blck_grp.csv', 'code_race': 'AHY2', 'code_ethnicity': 'AHZA', 'code_poverty': 'AH1J'}
Count of all CBGs in the US: 219772
Count of CBGs in the 10 states of interest: 40586
CBG table records: 220333
Count of CBGs in the study area after join: 40586
Compute race and ethnicity shares
Compute poverty shares
Saving to a file

Processing 2018_2022 {'boundary_filepath': 'US_blck_grp_2022.shp', 'table_filepath': 'nhgis0056_ds262_20225_blck_grp.csv', 'code_race': 'AQNG', 'code_ethnicity': 'AQNO', 'code_poverty': 'AQPZ'}
Count of all CBGs in the US: 241776
Count of CBGs in the 10 states of interest: 44174
CBG table records: 242336
Count of CBGs in the study